# Account Classification — v2

## Two strategies to handle the inner-transaction data problem

**Strategy A — Outer-only query (`is_inner = 0`)**  
- Query filters `is_inner = 0` at BigQuery level — no truncated inner groups  
- Drops columns irrelevant without inners: `block_number`, `hash`, `clawback_from_addr_id`, `token_cfg`, `token_unit_name`  
- Drops `to_addr_id / unique_senders_count` — receiving-side counts are meaningless without inner transfers  
- Gas flag: keeps `priority` and `rare_contract`; **removes** `congested_network`  

**Strategy B — Full query, narrow time window 
- Fetches ALL transactions (outer + inner) over a short `block_timestamp` range  
- Within 5 min every block 5*60/3 for 100 blocks   with 40 transaction with block, inners can have more than 100 inners so for 40 transactions 4000k per block, might need to recover blocks individually but then could have empty blocks, aim to gain more inners information even.
- All features including inner-dependent ones (`inner_token_unique_count`, `unique_senders_count`, `gas_congestion`) are valid  

In [1]:
import os, json, warnings, time, requests
import numpy as np
import pandas as pd
from datetime import datetime, timedelta, timezone
from google.cloud import bigquery
from google.oauth2 import service_account
from dotenv import load_dotenv
import matplotlib.pyplot as plt
import seaborn as sns


warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 60)

load_dotenv()
PROJECT_ID = os.getenv('BQ_PROJECT_ID')
print(f'Project: {PROJECT_ID}')

Project: sturdy-actor-486511-r4


In [ ]:
credentials = service_account.Credentials.from_service_account_file(
    filename='credentials/credentials.json'
)
client = bigquery.Client(credentials=credentials, project=PROJECT_ID)

---
# STRATEGY A — Outer-only (`is_inner = 0`)
Query excludes inner transactions entirely.  
Columns and features that depend on inners are dropped.

In [3]:
# ── Columns excluded from query (not needed without inner context) ─────────────
# block_number          used only to link inners to root tx
# hash                  tx hash — not a behavioural feature
# clawback_from_addr_id rarely populated, inner-related
# token_cfg             token metadata, not a feature
# token_unit_name       token metadata, not a feature

def load_outer_only(
    start_date: datetime,
    end_date: datetime,
    daily_limit: int = 5000
) -> pd.DataFrame:
    """
    Strategy A: query outer transactions only (is_inner = false).
    Excluded columns: block_number, hash, clawback_from_addr_id, token_cfg, token_unit_name.
    """
    all_dfs = []
    current = start_date
    while current <= end_date:
        date_str = current.strftime('%Y-%m-%d')
        query = f"""
            SELECT * EXCEPT(
                block_number, transaction_hash,
                clawback_from_addr_id, token_cfg, token_unit_name
            )
            FROM `nodely-data-pub.algorand_mainnet_daily.transactions`
            WHERE DATE(block_timestamp) = '{date_str}'
              AND is_inner = false
            LIMIT {daily_limit}
        """
        try:
            df_day = client.query(query).to_dataframe()
            all_dfs.append(df_day)
            print(f'  ok {date_str}: {len(df_day):,} rows (outer only)')
        except Exception as e:
            print(f'  fail {date_str}: {e}')
        current += timedelta(days=1)

    if not all_dfs:
        raise RuntimeError('No data loaded — all queries failed. Check credentials and date range.')

    df = pd.concat(all_dfs, ignore_index=True)
    print(f'\nStrategy A — total rows: {len(df):,}')
    return df


START_A = datetime(2025, 10, 3)
END_A   = datetime(2025, 10, 4)
df_a = load_outer_only(START_A, END_A)

  ok 2025-10-03: 5,000 rows (outer only)
�,Mt���B<�]���|<=�=#�L|�\"ͣ��\<���#L�\=C||]�L���\<�#��]�bkck�l�K    failed

Strategy A — total rows: 5,000


In [4]:
df_a

,block_timestamp,transaction_group,transaction_offset,transaction_type,from_addr_id,to_addr_id,value_float,value_int,close_to_addr_id,close_value_float,close_value_int,gas,note,rekey_to_addr_id,auth_by_addr_id,has_inners,is_inner,token_id,token_total,contract_id,input,root_offset,parent_offset,depth,inners_all,inners_direct,deltas
0,2025-10-03 04:43:45+00:00,None,40,app_call,8714958629,<NA>,0.0,0E-9,<NA>,0.0,0E-9,1000,None,<NA>,<NA>,False,False,<NA>,None,2982382116,"{""apaa"": [""+9vxEQ=="", ""AA==""], ""apid"": 2982382116}",0,<NA>,0,<NA>,0,"{""gd"": {""b3V0Y29tZQ=="": {""at"": 2}, ""aXNfcmVzb2x2ZWQ="": {..."
1,2025-10-03 03:34:44+00:00,None,10,app_call,8714958629,<NA>,0.0,0E-9,<NA>,0.0,0E-9,1000,None,<NA>,<NA>,False,False,<NA>,None,2982382898,"{""apaa"": [""+9vxEQ=="", ""AA==""], ""apid"": 2982382898}",0,<NA>,0,<NA>,0,"{""gd"": {""b3V0Y29tZQ=="": {""at"": 2}, ""aXNfcmVzb2x2ZWQ="": {..."
2,2025-10-03 04:40:02+00:00,None,35,app_call,8714958629,<NA>,0.0,0E-9,<NA>,0.0,0E-9,1000,None,<NA>,<NA>,False,False,<NA>,None,3197203697,"{""apaa"": [""+9vxEQ=="", ""AA==""], ""apid"": 3197203697}",0,<NA>,0,<NA>,0,"{""lg"": [""FR98dQE=""]}"
3,2025-10-03 03:43:22+00:00,None,13,app_call,8714958629,<NA>,0.0,0E-9,<NA>,0.0,0E-9,1000,None,<NA>,<NA>,False,False,<NA>,None,3240924086,"{""apaa"": [""+9vxEQ=="", ""AA==""], ""apid"": 3240924086}",0,<NA>,0,<NA>,0,"{""gd"": {""b3V0Y29tZQ=="": {""at"": 2}, ""aXNfcmVzb2x2ZWQ="": {..."
4,2025-10-03 03:43:22+00:00,None,11,app_call,8714958629,<NA>,0.0,0E-9,<NA>,0.0,0E-9,1000,None,<NA>,<NA>,False,False,<NA>,None,3241099385,"{""apaa"": [""+9vxEQ=="", ""AA==""], ""apid"": 3241099385}",0,<NA>,0,<NA>,0,"{""gd"": {""b3V0Y29tZQ=="": {""at"": 2}, ""aXNfcmVzb2x2ZWQ="": {..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4995,2025-10-03 00:27:29+00:00,b'\xd7$\x93o|2\xaf\xf7b\xf1I\xf0\xcc\xaa\xac=\x92\xa17\x...,54,app_call,8879667549,<NA>,0.0,0E-9,<NA>,0.0,0E-9,1000,None,<NA>,<NA>,True,False,<NA>,None,3241341203,"{""apaa"": [""4RryoA=="", ""AAAAAAAFfkA="", ""AAAAAGNXUKo="", ""A...",54,<NA>,0,7,4,"{""lg"": [""FR98dQAAAADBQzs7""]}"
4996,2025-10-03 03:24:12+00:00,b'\x847@}\x88\x91\xab\x12O\x1e\x11\x16+\xff\xaf5\xfc\x86...,9,app_call,8879667549,<NA>,0.0,0E-9,<NA>,0.0,0E-9,1000,None,<NA>,<NA>,True,False,<NA>,None,3241351970,"{""apaa"": [""4RryoA=="", ""AAAAAAAFfkA="", ""AAAAAGNXUKo="", ""A...",9,<NA>,0,7,4,"{""lg"": [""FR98dQAAAADBRdnI""]}"
4997,2025-10-03 00:45:56+00:00,b'-j\x10\xe7\xbf\x88\xd3W\x8d1\xbcB\xc8%(%Y\x82\xccJuX9\...,2,app_call,8879667549,<NA>,0.0,0E-9,<NA>,0.0,0E-9,1000,None,<NA>,<NA>,True,False,<NA>,None,3241397728,"{""apaa"": [""4RryoA=="", ""AAAAAAAFfkA="", ""AAAAAGNXUKo="", ""A...",2,<NA>,0,7,4,"{""lg"": [""FR98dQAAAADBQ34m""]}"
4998,2025-10-03 01:05:04+00:00,b'\xbbz\xba\x97\x17\xab\xbd\x9e/\xad\xb8\xa0 ;6\xa0\xa0c...,30,app_call,8879667549,<NA>,0.0,0E-9,<NA>,0.0,0E-9,1000,None,<NA>,<NA>,True,False,<NA>,None,3241397728,"{""apaa"": [""4RryoA=="", ""AAAAAAAFfkA="", ""AAAAAGNXUKo="", ""A...",30,<NA>,0,7,4,"{""lg"": [""FR98dQAAAADBQ86W""]}"


In [5]:
# ── Base user table ────────────────────────────────────────────────────────────
df_users_a = (
    pd.DataFrame({'from_addr_id': df_a['from_addr_id'].dropna().unique()})
    .reset_index(drop=True)
)
df_users_a['tx_count'] = df_users_a['from_addr_id'].map(
    df_a.groupby('from_addr_id').size()
)
print(f'Unique accounts (A): {len(df_users_a):,}')

df_users_a

Unique accounts (A): 297


,from_addr_id,tx_count
0,8714958629,74
1,8630484703,4088
2,1189090998,1
3,479050446,3
4,1527083747,15
...,...,...
292,674930967,1
293,1056124101,1
294,1163379515,1
295,1390806783,1


By focusing on Outers we get more data relevant for the decision making of the tx .

We remove inners so automated recalls dont inflate real user interactions.

Further below.
By focusing on more inners, this gives more details on the traceback of the group transaction.
Giving more information on the initial user.

Giving a structure for each contract also could be usefull to see how each contract can be intereacted with.

In [6]:
# ── contract_unique_count & token_unique_count (outer txs only) ────────────────
for col_in, col_out in [('contract_id', 'contract_unique_count'), ('token_id', 'token_unique_count')]:
    counts = df_a.groupby('from_addr_id')[col_in].nunique().rename(col_out)
    df_users_a = (
        df_users_a
        .merge(counts, on='from_addr_id', how='left')
        .fillna({col_out: 0})
        .astype({col_out: int})
    )
df_users_a.sort_values('tx_count', ascending=False).head()

,from_addr_id,tx_count,contract_unique_count,token_unique_count
1,8630484703,4088,1,0
273,8879667549,388,62,0
0,8714958629,74,70,0
69,981903663,22,1,0
7,5906347936,21,1,0


contract unique count 
this expects to find the variability of interactions
Real users are expeccted to interact with more contracts on average than bots 


# Account Classification Matrix

|                          | High Interactions | Low Interactions |
|:------------------------:|:-----------------:|:----------------:|
| **High Contract Amount** | `1` — Bot         | `2` — Bot        |
| **Low Contract Amount**  | `3` — Uncertain   | `4` — Human      |

In [7]:
# ── Gas flag — Strategy A ──────────────────────────────────────────────────────
# gas_flag values:
#   0  = normal fee OR rare contract (<=5 unique users)
#   1  = real priority bid  → bot signal
#   removed chain congestion. 

MIN_FEE = 1000
df_a['inners_all_n'] = pd.to_numeric(df_a['inners_all'], errors='coerce').fillna(0)
df_a['gas']          = pd.to_numeric(df_a['gas'],        errors='coerce').fillna(MIN_FEE)
df_a['_expected_gas'] = (1 + df_a['inners_all_n']) * MIN_FEE

contract_users_a = df_a.groupby('contract_id')['from_addr_id'].nunique().rename('_cu')
df_a = df_a.merge(contract_users_a, on='contract_id', how='left')
df_a['_cu'] = df_a['_cu'].fillna(1)

def gas_flag_a(row):
    if row['gas'] <= row['_expected_gas']:  return 0   # normal
    if row['_cu'] <= 5:                     return -1   # rare contract
    return 1                                           # priority bid

df_a['gas_flag'] = df_a.apply(gas_flag_a, axis=1)
print('gas_flag (A):', df_a['gas_flag'].value_counts().to_dict())

gas_agg_a = (
    df_a.groupby('from_addr_id')['gas_flag']
    .agg(gas_priority_count=lambda x: (x==1).sum(),
         gas_priority_ratio =lambda x: (x==1).mean())
    .reset_index()
)
df_users_a = (
    df_users_a
    .merge(gas_agg_a, on='from_addr_id', how='left')
    .fillna({'gas_priority_count': 0, 'gas_priority_ratio': 0.0})
    .astype({'gas_priority_count': int})
)

df_users_a

gas_flag (A): {0: 4976, -1: 24}


,from_addr_id,tx_count,contract_unique_count,token_unique_count,gas_priority_count,gas_priority_ratio
0,8714958629,74,70,0,0,0.0
1,8630484703,4088,1,0,0,0.0
2,1189090998,1,1,0,0,0.0
3,479050446,3,1,0,0,0.0
4,1527083747,15,1,0,0,0.0
...,...,...,...,...,...,...
292,674930967,1,1,0,0,0.0
293,1056124101,1,1,0,0,0.0
294,1163379515,1,1,0,0,0.0
295,1390806783,1,1,0,0,0.0


In [8]:
df_users_a

,from_addr_id,tx_count,contract_unique_count,token_unique_count,gas_priority_count,gas_priority_ratio
0,8714958629,74,70,0,0,0.0
1,8630484703,4088,1,0,0,0.0
2,1189090998,1,1,0,0,0.0
3,479050446,3,1,0,0,0.0
4,1527083747,15,1,0,0,0.0
...,...,...,...,...,...,...
292,674930967,1,1,0,0,0.0
293,1056124101,1,1,0,0,0.0
294,1163379515,1,1,0,0,0.0
295,1390806783,1,1,0,0,0.0


Gas consumption is recomended to be payed at the outher of the transaction.

Gas consumption can be payed by any account in the transaction.

Gas overpayments 

0 no overpay

1 overpay

-1 overpay but flag it

Checking gas consumptions from the main df.
This iteration doesnt find much weird consumptions.
I expected low amounts over 1000 
So either minimal gas is always used
Contracts might run in a bad loop
And the 9000 extra gas consumption case.

In [9]:
df_a['gas'].value_counts()

gas
1000     4605
0         197
2000      102
3000       72
8000       11
10000      11
6000        2
Name: count, dtype: Int64

In [10]:
# ── Note features ──────────────────────────────────────────────────────────────
df_a['_ns'] = df_a['note'].fillna('').astype(str)
df_a['_nr'] = df_a.groupby('from_addr_id')['_ns'].transform(
    lambda x: (x[x != ''].duplicated(keep=False).any() if (x != '').any() else False)
).astype(int)
df_a['_na'] = ((df_a['_ns'].str.len() > 8) & (~df_a['_ns'].str.contains(' ', na=False)) & (df_a['_ns'] != '')).astype(int)

note_agg_a = (
    df_a.groupby('from_addr_id')
    .agg(note_repetitive=('_nr', 'max'),
         note_autogen_ratio=('_na', 'mean'),
         note_has_content=('_ns', lambda x: int((x != '').any())))
    .reset_index()
)
df_users_a = df_users_a.merge(note_agg_a, on='from_addr_id', how='left')
df_users_a['note_repetitive']    = df_users_a['note_repetitive'].fillna(0).astype(int)
df_users_a['note_autogen_ratio'] = df_users_a['note_autogen_ratio'].fillna(0.0)
df_users_a['note_has_content']   = df_users_a['note_has_content'].fillna(0).astype(int)

In [11]:
df_users_a

# note repetitive 
# Could indicate either a commoon wallet sign
# a common interaction with the same patern

#note autogen attempts to catch long strings of data uncomon for humans
# this could be contract automated responses
# someone keepeng data in blockchain
# note has content is to check if it has data & none of the above

,from_addr_id,tx_count,contract_unique_count,token_unique_count,gas_priority_count,gas_priority_ratio,note_repetitive,note_autogen_ratio,note_has_content
0,8714958629,74,70,0,0,0.0,0,0.0,0
1,8630484703,4088,1,0,0,0.0,1,0.0,1
2,1189090998,1,1,0,0,0.0,0,0.0,0
3,479050446,3,1,0,0,0.0,0,0.0,0
4,1527083747,15,1,0,0,0.0,0,0.0,0
...,...,...,...,...,...,...,...,...,...
292,674930967,1,1,0,0,0.0,0,0.0,0
293,1056124101,1,1,0,0,0.0,0,0.0,0
294,1163379515,1,1,0,0,0.0,0,0.0,0
295,1390806783,1,1,0,0,0.0,0,0.0,0


In [12]:
# ── Rekey, Fleet, Accounts controlled / rekeyed to ────────────────────────────
rekey_a = df_a.groupby('from_addr_id')['rekey_to_addr_id'].agg(has_rekeyed=lambda x: int(x.notna().any())).reset_index()

fleet_a = (df_a[df_a['auth_by_addr_id'].notna()]
           .groupby('auth_by_addr_id')['from_addr_id'].nunique().rename('controller_fleet').reset_index())
ac_a = (df_a[df_a['auth_by_addr_id'].notna()][['from_addr_id','auth_by_addr_id']].drop_duplicates('from_addr_id'))
fleet_per_a = ac_a.merge(fleet_a, on='auth_by_addr_id', how='left')

ctrl_a = (df_a[df_a['auth_by_addr_id'].notna()]
          .groupby('auth_by_addr_id')['from_addr_id'].nunique().rename('accounts_controlled').reset_index()
          .rename(columns={'auth_by_addr_id':'from_addr_id'}))
rkt_a  = (df_a[df_a['rekey_to_addr_id'].notna()]
          .groupby('rekey_to_addr_id')['from_addr_id'].nunique().rename('accounts_rekeyed_to').reset_index()
          .rename(columns={'rekey_to_addr_id':'from_addr_id'}))

df_users_a = (
    df_users_a
    .merge(rekey_a, on='from_addr_id', how='left')
    .merge(fleet_per_a[['from_addr_id','controller_fleet']], on='from_addr_id', how='left')
    .merge(ctrl_a, on='from_addr_id', how='left')
    .merge(rkt_a,  on='from_addr_id', how='left')
)
for c in ['has_rekeyed','controller_fleet','accounts_controlled','accounts_rekeyed_to']:
    df_users_a[c] = df_users_a[c].fillna(0).astype(int)

multising in algorand works with  account not rekey.

Rekey alows another account to sign for that account.

Users could rekey or auth accounts for privacy and security.

So while rekey should not be that uncomon for regular users. Its mainly used by big accounts.

Auth account is more related to bots.

By checking the amount of accounts that delegate the key or the auth to the same accounts we can get more context on the purpose of the reky or auth

running into the problem of the inners hidding info at some of those 

In [13]:
# ── Timing features ────────────────────────────────────────────────────────────
def compute_min_gap(x):
    d = x.diff().dt.total_seconds().dropna()
    if len(d) == 0: return -1.0
    return 0.1 if d.min() == 0.0 else d.min()

def has_periodic_gap(x):
    d = x.diff().dt.total_seconds().dropna().round(0)
    return 1 if len(d) >= 2 and d.value_counts().max() >= 3 else 0

sa = df_a.sort_values(['from_addr_id','block_timestamp'])
mg_a = sa.groupby('from_addr_id')['block_timestamp'].apply(compute_min_gap).rename('timing_min_gap_s').reset_index()
prd_a = sa.groupby('from_addr_id')['block_timestamp'].apply(has_periodic_gap).rename('is_periodic').reset_index()

df_users_a = df_users_a.merge(mg_a, on='from_addr_id', how='left').merge(prd_a, on='from_addr_id', how='left')
df_users_a['timing_min_gap_s'] = df_users_a['timing_min_gap_s'].fillna(-1.0)
df_users_a['is_periodic']      = df_users_a['is_periodic'].fillna(0).astype(int)

Bot features we are considering : 
High frequency
Constant
Oportunistic.

2 transactions spaced on the same time is odd, increasing to 3 to make it more flagrant.
but since its automated could be 10.

min time stays between outers counts



In [14]:
# ── Value in ALGO (shared helpers) ────────────────────────────────────────────
SUPPORTED_ASSETS = {
    'usdc':31566704,'hog':3178895177,'ora':1284444444,'talgo':2537013734,
    'vote':452399768,'gobtc':386192725,'xusd':760037151,'tiny':2200000000,
    'alpha':2726252423,'opul':287867876,'finite':400593267,'niko':1265975021,
    'pow':2994233666,'xalgo':1134696561,'coop':796425061,'vest':700965019,
    'goeth':386195940,'compx':1732165149,'usdt':312769,'sparky':3054226103
}
SUPPORTED_IDS    = set(SUPPORTED_ASSETS.values())
ASSET_NATIVE     = 971381860
ASSET_USDC       = 971384592
WC_ASSETS        = [127746157,127745593,127746786,3145862805]
ASSET_USDC_SC    = 849191641
USDC_SF          = 0.01484306067
ALGO_CAP         = 1_000_000
_pcache          = {}

def _unix(s, e):
    s = datetime.strptime(s,'%Y-%m-%d').replace(tzinfo=timezone.utc)
    e = datetime.strptime(e,'%Y-%m-%d').replace(tzinfo=timezone.utc) + timedelta(days=1) - timedelta(seconds=1)
    return int(s.timestamp()), int(e.timestamp())

def get_price(aid, sd, ed):
    su, eu = _unix(sd, ed)
    url = f'https://indexer.vestige.fi/assets/{aid}/candles?network_id=0&interval=86400&start={su}&end={eu}&denominating_asset_id=0&volume_in_denominating_asset=false'
    try:
        r = requests.get(url, timeout=10); r.raise_for_status()
        data = r.json()
        cp = [c['close'] for c in data if c.get('close') is not None]
        return sum(cp)/len(cp) if cp else None
    except: return None

def fetch_prices(sd, ed):
    _pcache.clear()
    for name, aid in SUPPORTED_ASSETS.items():
        _pcache[aid] = get_price(aid, sd, ed); time.sleep(0.5)
    print(f'Prices fetched: {sum(1 for v in _pcache.values() if v is not None)}/{len(SUPPORTED_ASSETS)}')

def tx_value(row):
    tid = row['token_id']; val = float(row['value_float']) if pd.notna(row['value_float']) else 0.0
    if pd.isna(tid): return min(val, ALGO_CAP)
    tid = int(tid)
    if tid == ASSET_NATIVE: return min(val, ALGO_CAP)
    if tid == ASSET_USDC:
        p = _pcache.get(SUPPORTED_ASSETS['usdc']); return min(val*p, ALGO_CAP) if p else 0.0
    if tid == ASSET_USDC_SC:
        p = _pcache.get(SUPPORTED_ASSETS['usdc']); return min(val*p*USDC_SF, ALGO_CAP) if p else 0.0
    if tid in WC_ASSETS: return min(val, ALGO_CAP)
    if tid in SUPPORTED_IDS:
        p = _pcache.get(tid); return min(val*p, ALGO_CAP) if p else 0.0
    return 0.0

fetch_prices(START_A.strftime('%Y-%m-%d'), END_A.strftime('%Y-%m-%d'))
df_a['value_algo'] = df_a.apply(tx_value, axis=1)

val_a = df_a.groupby('from_addr_id')['value_algo'].sum().rename('value_algo_cumulative').reset_index()
df_users_a = df_users_a.merge(val_a, on='from_addr_id', how='left')
df_users_a['value_algo_cumulative'] = df_users_a['value_algo_cumulative'].fillna(0.0)
print(f'df_users_a final shape: {df_users_a.shape}')
df_users_a['value_algo_cumulative'].value_counts() #value added on same currency but is it algo or other ?

Prices fetched: 20/20
df_users_a final shape: (297, 16)


value_algo_cumulative
0.0    297
Name: count, dtype: int64

In [15]:
df_users_a['value_algo_cumulative'].value_counts()

value_algo_cumulative
0.0    297
Name: count, dtype: int64

In [16]:
# ── Strategy A — Classification ────────────────────────────────────────────────
from sklearn.ensemble import IsolationForest
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.preprocessing import StandardScaler

# to_addr_id / unique_senders_count EXCLUDED — meaningless without inners
FEATURES_A = [
    'tx_count','contract_unique_count','token_unique_count',
    'gas_priority_ratio','note_autogen_ratio',
    'accounts_controlled','accounts_rekeyed_to',
    'timing_min_gap_s','is_periodic','value_algo_cumulative'
]

Xa = df_users_a[FEATURES_A].copy()
Xa['timing_min_gap_s'] = Xa['timing_min_gap_s'].replace(-1.0, 0)
Xa['timing_bot_signal'] = 1 / (Xa['timing_min_gap_s'] + 1)
for c in ['tx_count','value_algo_cumulative']: Xa[c] = np.log1p(Xa[c])

sc_a = StandardScaler()
Xa_s = sc_a.fit_transform(Xa)

df_users_a['iso_bot'] = (IsolationForest(contamination=0.2, random_state=42).fit_predict(Xa_s) == 1).astype(int)

Xa_n = (Xa - Xa.min()) / (Xa.max() - Xa.min() + 1e-9)
w_a = {'tx_count':0.30,'timing_bot_signal':0.25,'is_periodic':0.20,
       'contract_unique_count':0.10,'accounts_controlled':0.08,
       'value_algo_cumulative':0.05,'accounts_rekeyed_to':0.02}
df_users_a['bot_score']   = sum(Xa_n[c]*w for c,w in w_a.items())
df_users_a['weighted_bot']= (df_users_a['bot_score'] > 0.15).astype(int)

dt_a = DecisionTreeClassifier(max_depth=4, random_state=42)
dt_a.fit(Xa_s, df_users_a['weighted_bot'])
df_users_a['dt_bot'] = dt_a.predict(Xa_s)

print('=== Strategy A Results ===')
print('Isolation Forest:', df_users_a['iso_bot'].value_counts().to_dict())
print('Weighted Score:  ', df_users_a['weighted_bot'].value_counts().to_dict())
print('Decision Tree:   ', df_users_a['dt_bot'].value_counts().to_dict())
print('\nRules:')
print(export_text(dt_a, feature_names=list(Xa.columns)))
print('\nAgreement:')
print(df_users_a[['iso_bot','weighted_bot','dt_bot']].value_counts())

=== Strategy A Results ===
Isolation Forest: {1: 272, 0: 25}
Weighted Score:   {1: 279, 0: 18}
Decision Tree:    {1: 279, 0: 18}

Rules:
|--- timing_min_gap_s <= -0.08
|   |--- class: 1
|--- timing_min_gap_s >  -0.08
|   |--- is_periodic <= 3.14
|   |   |--- class: 0
|   |--- is_periodic >  3.14
|   |   |--- class: 1


Agreement:
iso_bot  weighted_bot  dt_bot
1        1             1         272
0        0             0          18
         1             1           7
Name: count, dtype: int64


In [17]:
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
import numpy as np

# ── Features disponibles en Strategy A ────────────────────────────────────────
FEATURES_A = [
    'tx_count',
    'contract_unique_count',
    'token_unique_count',
    'gas_priority_ratio',
    'note_autogen_ratio',
    'accounts_controlled',
    'accounts_rekeyed_to',
    'controller_fleet',
    'timing_min_gap_s',
    'is_periodic',
    'value_algo_cumulative'
]

# ── Preparar matriz de features ───────────────────────────────────────────────
X = df_users_a[FEATURES_A].copy()
X['timing_min_gap_s'] = X['timing_min_gap_s'].replace(-1.0, 0)
X['timing_bot_signal'] = 1 / (X['timing_min_gap_s'] + 1)
X['tx_count']                = np.log1p(X['tx_count'])
X['value_algo_cumulative']   = np.log1p(X['value_algo_cumulative'])

# ── Isolation Forest ──────────────────────────────────────────────────────────
sc = StandardScaler()
X_s = sc.fit_transform(X)
iso = IsolationForest(contamination=0.2, random_state=42)
df_users_a['iso_score'] = iso.fit(X_s).decision_function(X_s)  # score continuo
df_users_a['iso_bot']   = (iso.fit_predict(X_s) == -1).astype(int)

# ── Weighted score ────────────────────────────────────────────────────────────
X_n = (X - X.min()) / (X.max() - X.min() + 1e-9)
weights = {
    'tx_count':             0.25,
    'timing_bot_signal':    0.20,
    'is_periodic':          0.15,
    'contract_unique_count':0.10,
    'accounts_controlled':  0.12,
    'controller_fleet':     0.10,
    'value_algo_cumulative':0.05,
    'accounts_rekeyed_to':  0.03,
}
df_users_a['bot_score'] = sum(X_n[c] * w for c, w in weights.items() if c in X_n.columns)

# ── Etiqueta final: Bot / Uncertain / Human ───────────────────────────────────
def classify(row):
    if row['iso_bot'] == 1 and row['bot_score'] > 0.20:
        return 'Bot'
    elif row['iso_bot'] == 0 and row['bot_score'] < 0.10:
        return 'Human'
    else:
        return 'Uncertain'

df_users_a['label'] = df_users_a.apply(classify, axis=1)
print(df_users_a['label'].value_counts())

label
Uncertain    283
Bot           14
Name: count, dtype: int64


In [18]:
def classify_account(addr: str) -> dict:
    row = df_users_a[df_users_a['from_addr_id'] == addr]
    
    if row.empty:
        return {'error': f'Account {addr} not found in dataset'}
    
    row = row.iloc[0]
    
    # razones principales
    reasons = []
    if row['accounts_controlled'] > 5:
        reasons.append(f"controls {int(row['accounts_controlled'])} accounts")
    if row['controller_fleet'] > 5:
        reasons.append(f"part of fleet of {int(row['controller_fleet'])}")
    if row['is_periodic'] == 1:
        reasons.append("periodic timing detected")
    if row['note_autogen_ratio'] > 0.5:
        reasons.append(f"autogenerated notes ({row['note_autogen_ratio']:.0%})")
    if row['tx_count'] > np.log1p(50):
        reasons.append(f"high tx volume")
    
    return {
        'address':   addr,
        'label':     row['label'],
        'bot_score': round(row['bot_score'], 3),
        'iso_score': round(row['iso_score'], 3),
        'reasons':   reasons if reasons else ['no strong signals']
    }

# ── Uso ───────────────────────────────────────────────────────────────────────
addr = df_users_a['from_addr_id'].iloc[0]  
result = classify_account(addr)
print(f"\nAddress : {result['address']}")
print(f"Label   : {result['label']}")
print(f"Score   : {result['bot_score']}")
print(f"Reasons : {', '.join(result['reasons'])}")


Address : 8714958629
Label   : Bot
Score   : 0.551
Reasons : periodic timing detected, high tx volume


---
# STRATEGY B — Full data, narrow time window (15–30 min)
Fetch ALL transactions (outer + inner) for a short `block_timestamp` range.  
Within 15–30 min every block is captured completely — inner groups are intact.

look a bit more onto this and go to the pdf .


In [19]:
# ── define window ─────────────────────────────────────────────────────────────
WINDOW_START   = datetime(2025, 10, 3, 12, 0, 0)
WINDOW_MINUTES = 30
WINDOW_END     = WINDOW_START + timedelta(minutes=WINDOW_MINUTES)

s_str = WINDOW_START.strftime('%Y-%m-%d %H:%M:%S')
e_str = WINDOW_END.strftime('%Y-%m-%d %H:%M:%S')

# ── Step 1: find 5 busy blocks with < 1000 transactions ──────────────────────
df_block_counts = client.query(f"""
    SELECT block_number, COUNT(*) AS tx_count
    FROM `nodely-data-pub.algorand_mainnet_daily.transactions`
    WHERE block_timestamp >= '{s_str}'
      AND block_timestamp <  '{e_str}'
    GROUP BY block_number
    HAVING tx_count < 1000
    ORDER BY tx_count DESC
    LIMIT 5
""").to_dataframe()

print('Top 5 busiest blocks under 1000 txs:')
print(df_block_counts)
top5_blocks = df_block_counts['block_number'].tolist()

Top 5 busiest blocks under 1000 txs:
   block_number  tx_count
0      54259375       810
1      54259371       426
2      54259373       402
3      54259447       390
4      54259374       246


In [ ]:
# ── Step 2: fetch all transactions for those 5 blocks ────────────────────────
blocks_sql = ', '.join(str(b) for b in top5_blocks)

job = client.query(f"""
    SELECT *
    FROM `nodely-data-pub.algorand_mainnet_daily.transactions`
    WHERE block_number IN ({blocks_sql})
""")

# create_bqstorage_client=False avoids the pyarrow conversion issue
df_b_raw = job.to_dataframe(create_bqstorage_client=False)

print(f'Rows loaded: {len(df_b_raw):,}')
print(df_b_raw['is_inner'].value_counts())

df_b_raw.to_parquet('outputs/df_b_raw.parquet', index=False)
print('Saved.')

df_b_raw.head()

Rows loaded: 2,274
is_inner
False    1826
True      448
Name: count, dtype: Int64
Saved.


,block_number,block_timestamp,transaction_hash,transaction_group,transaction_offset,transaction_type,from_addr_id,to_addr_id,value_float,value_int,close_to_addr_id,close_value_float,close_value_int,gas,note,rekey_to_addr_id,auth_by_addr_id,has_inners,is_inner,clawback_from_addr_id,token_cfg,token_id,token_total,token_unit_name,contract_id,input,root_offset,parent_offset,depth,inners_all,inners_direct,deltas
0,54259447,2025-10-03 12:03:47+00:00,EQCNCCVEQSML6QZLYT4CJZHPQLOEEO4RKMI6NTG6MZX65IFP7THA,b'\x0c\xb5\xa6\x8d\xbe\xcfM\x97g6S\xf9\x08\xe1B\x00E\xc2...,11,app_call,10111638208,<NA>,0.0,0E-9,<NA>,0.0,0E-9,3000,None,<NA>,<NA>,True,False,<NA>,{'list': []},<NA>,None,None,1027956681,"{""apaa"": [""/95jeA=="", ""AAAAAAAAAAI="", ""AAAAAAAAAAA=""], ""...",11,<NA>,0,2,2,{}
1,54259374,2025-10-03 12:00:22+00:00,VWTDL2CWIFE2EASP3W2X3PH4JYSIDRW46QCV35MGO5FQFN6QBEOA,None,234,app_call,8630484703,<NA>,0.0,0E-9,<NA>,0.0,0E-9,1000,1,<NA>,<NA>,False,False,<NA>,{'list': []},<NA>,None,None,2773418797,"{""apaa"": [""2wSR/w=="", ""AAAAAAACDr8=""], ""apas"": [12844444...",0,<NA>,0,<NA>,0,"{""lg"": [""RQ==""]}"
2,54259371,2025-10-03 12:00:13+00:00,AYLJDRY5QUIAQ57KOLJQ246UAQKQF5LG6TIATHQQQAJTQ3I74HQQ,b'\x91\x80\x80\xe1\xfaS\x86\x9d[\xdfI\x15$\\\x1be\xb6\x8...,242,app_call,10525569002,<NA>,0.0,0E-9,<NA>,0.0,0E-9,0,deflex-hay-0.7881903959473415,<NA>,<NA>,True,False,<NA>,{'list': []},<NA>,None,None,3172554435,"{""apaa"": [""3NM24w=="", ""AAAAAAABo1Y="", ""AAAAAAAAAAY=""], ""...",242,<NA>,0,5,5,"{""gd"": {""ZXhwZWN0ZWRfYW1vdW50X291dA=="": {""at"": 2, ""ui"": ..."
3,54259375,2025-10-03 12:00:25+00:00,BUZWZT5MV6UOLI7SCHD5XKCTHKC4PAZYVXOYP6H4Y6TCGUD7CQ5A,b'7\xb5&\xc7/\xba\x96]Yo\xa9\x92\xd3P\xd0\x87Bi!/\x01\x1...,701,app_call,10516118256,<NA>,0.0,0E-9,<NA>,0.0,0E-9,1000,None,<NA>,<NA>,True,False,<NA>,{'list': []},<NA>,None,None,3241007779,"{""apaa"": [""4RryoA=="", ""AAAAAAAHoSA="", ""AAAAAACYloA="", ""A...",701,<NA>,0,7,4,"{""lg"": [""FR98dQAAAADBS/oW""]}"
4,54259371,2025-10-03 12:00:13+00:00,HQH4DYVNVLJAG3NG7E3B4CM2HP524QWNYRIWOCEHYT5JX4UP6KFQ,None,161,app_call,2656598227,<NA>,0.0,0E-9,<NA>,0.0,0E-9,0,None,<NA>,<NA>,False,True,<NA>,{'list': []},<NA>,None,None,2611131944,"{""apaa"": [""7NHaOA=="", ""AQ==""], ""apfa"": [971350278], ""api...",160,160,1,1,1,"{""gd"": {""aQ=="": {""at"": 1, ""bs"": ""AAONfqTGgAAAAAkYTnKgAAA..."


In [ ]:
# ── Link inners to their root account ─────────────────────────────────────────
df_inners_b = (
    inner_b.merge(
        outer_b[['block_number','transaction_offset','from_addr_id']],
        left_on=['block_number','root_offset'],
        right_on=['block_number','transaction_offset'],
        how='inner', suffixes=('_inner','_user')
    )
)
print(f'Inners linked: {len(df_inners_b):,}')